# QRAM Tutorial - Bucket-Brigade Circuit And Simulation

This notebook introduces a compact QRAM workflow built around a bucket-brigade
circuit. A small executable example demonstrates the mechanism end to end, and
checked-in resource data is used to show how the requirements scale with QRAM
level.

The main example is deliberately small:

| Tutorial item | Scale used here | Why |
| --- | --- | --- |
| address register | 2 qubits | covers four memory cells: `00`, `01`, `10`, `11` |
| data cells | 4 classical bits | enough to verify address-dependent readout |
| bus register | 1 qubit | one-bit QRAM readout |
| router/data ancillas | generated by `buckdatacell.Qram` | exposes the bucket-brigade tree |
| core validation | local `Statevector` | exact readout probabilities without GPU |
| optional noise demo | CPU `Qiskit Aer` shots | illustrative noisy readout, not calibrated hardware |
| processed experiment plots | project-included processed artifacts | visualize fidelity and mitigation trends |

The notebook does not reproduce the large calibrated noise simulations from the
project. Its goal is mechanism-level reproducibility: build a QRAM circuit,
verify the readout logic, compare CSWAP implementations, run a lightweight
noise demonstration, inspect level-by-level resource trends, and visualize
processed experimental datasets stored with the repository.


In [ ]:
%matplotlib inline

import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import qiskit
from IPython.display import display
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Statevector
from typing import Optional


def discover_repo_root(start: Path) -> Path:
    anchor = start.resolve()
    for candidate in (anchor, *anchor.parents):
        if (candidate / "qram").is_dir() and (candidate / "realdata").is_dir():
            return candidate
        if (candidate / "QRAM" / "qram").is_dir() and (candidate / "QRAM" / "realdata").is_dir():
            return candidate / "QRAM"
    raise FileNotFoundError("Could not locate the QRAM repository root.")


repo_root = discover_repo_root(Path.cwd())
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


def ensure_qiskit_legacy_register_fields() -> None:
    # The original QRAM source was written against an older Qiskit API.
    if not hasattr(QuantumRegister, "_size"):
        QuantumRegister._size = property(lambda self: self.size)
    if not hasattr(ClassicalRegister, "_size"):
        ClassicalRegister._size = property(lambda self: self.size)


ensure_qiskit_legacy_register_fields()
warnings.filterwarnings("ignore", category=DeprecationWarning)

from qram.config import Config
from qram.qramtemplate.buckdatacell import (
    Qram as BucketBrigadeQram,
    cswap_depth,
    swap_depth,
)


def format_table(headers, rows):
    rows = [[str(item) for item in row] for row in rows]
    widths = [
        max(len(str(header)), *(len(row[idx]) for row in rows)) if rows else len(str(header))
        for idx, header in enumerate(headers)
    ]
    header = " | ".join(str(value).ljust(widths[idx]) for idx, value in enumerate(headers))
    rule = "-+-".join("-" * width for width in widths)
    body = [" | ".join(value.ljust(widths[idx]) for idx, value in enumerate(row)) for row in rows]
    return "\n".join([header, rule, *body])


def address_labels(n_address_qubits: int) -> list[str]:
    return [format(i, f"0{n_address_qubits}b") for i in range(2**n_address_qubits)]


def prepare_address_register(circuit: QuantumCircuit, address_qreg: QuantumRegister, state: str) -> None:
    if state == "uniform":
        for qubit in address_qreg:
            circuit.h(qubit)
        return

    if len(state) != len(address_qreg) or any(bit not in "01" for bit in state):
        raise ValueError("state must be 'uniform' or a bit string matching the address width.")

    for idx, bit in enumerate(state):
        if bit == "1":
            circuit.x(address_qreg[idx])


def build_bucket_qram(
    n_address_qubits: int,
    data: list[int],
    *,
    mode: str = "native",
    address_state: str = "uniform",
    include_measurements: bool = False,
    load_bus: bool = True,
):
    addresses = address_labels(n_address_qubits)
    if len(data) != len(addresses):
        raise ValueError("data length must equal 2 ** n_address_qubits")

    address_qreg = QuantumRegister(n_address_qubits, "address")
    bus_qreg = QuantumRegister(1, "bus")
    if include_measurements:
        c_address = ClassicalRegister(n_address_qubits, "c_address")
        c_bus = ClassicalRegister(1, "c_bus")
        circuit = QuantumCircuit(address_qreg, bus_qreg, c_address, c_bus)
    else:
        circuit = QuantumCircuit(address_qreg, bus_qreg)

    prepare_address_register(circuit, address_qreg, address_state)

    config = Config()
    config.decompose_mode = mode
    config.load_bus = load_bus
    config.dcswap_embedding = False

    qram = BucketBrigadeQram(addresses, data, bandwidth=1, config=config)
    qram(circuit, address_qreg, bus_qreg)

    if include_measurements:
        circuit.measure(bus_qreg, c_bus)
        circuit.measure(address_qreg, c_address)

    return circuit, qram, address_qreg, bus_qreg


def logical_readout_probabilities(
    circuit: QuantumCircuit,
    address_qreg: QuantumRegister,
    bus_qreg: QuantumRegister,
    *,
    threshold: float = 1e-10,
) -> pd.DataFrame:
    qargs = [circuit.find_bit(q).index for q in [*address_qreg, *bus_qreg]]
    raw = Statevector.from_instruction(circuit).probabilities_dict(qargs=qargs)

    rows = []
    for bitstring, probability in raw.items():
        # Qiskit displays classical bit strings with the highest-index bit first.
        # Reversing here makes the table read as address[0], address[1], ..., bus[0].
        logical_bits = str(bitstring)[::-1]
        rows.append(
            {
                "address": logical_bits[: len(address_qreg)],
                "bus": logical_bits[len(address_qreg) : len(address_qreg) + len(bus_qreg)],
                "probability": float(probability),
            }
        )

    df = pd.DataFrame(rows).groupby(["address", "bus"], as_index=False)["probability"].sum()
    df = df[df["probability"] > threshold]
    return df.sort_values(["address", "bus"]).reset_index(drop=True)


def expected_table(addresses: list[str], data: list[int], probability: Optional[float] = None) -> pd.DataFrame:
    table = pd.DataFrame({"address": addresses, "expected_bus": [str(bit) for bit in data]})
    if probability is not None:
        table["expected_probability"] = probability
    return table


def resource_summary(circuit: QuantumCircuit) -> dict:
    return {
        "qubits": circuit.num_qubits,
        "clbits": circuit.num_clbits,
        "depth": circuit.depth(),
        "size": circuit.size(),
        "swap_depth": swap_depth(circuit),
        "cswap_or_cz_depth": cswap_depth(circuit),
        "ops": dict(circuit.count_ops()),
    }


print(
    format_table(
        ["item", "value"],
        [
            ["repo", str(repo_root)],
            ["qiskit", qiskit.__version__],
            ["core simulator", "qiskit.quantum_info.Statevector"],
            ["optional noisy simulator", "qiskit_aer.AerSimulator"],
            ["tutorial scale", "2 address qubits, 4 cells, 1 bus qubit"],
        ],
    )
)


## 1. Define A Small QRAM Instance

QRAM implements address-dependent data readout as a quantum circuit. This
tutorial uses 2 address qubits, which represent 4 memory cells, and stores a
small Boolean table:

`00 -> 0`, `01 -> 1`, `10 -> 1`, `11 -> 0`

This data pattern is useful for a tutorial because it is not all-zero or
all-one, so the readout must depend on the address. The circuit is still small
enough to verify each address directly with a statevector simulation.


In [ ]:
n_address_qubits = 2
memory_data = [0, 1, 1, 0]
addresses = address_labels(n_address_qubits)

memory_table = expected_table(addresses, memory_data)
print("Memory table used by the QRAM example")
display(memory_table)


## 2. Build The Bucket-Brigade Circuit

The bucket-brigade implementation used here lives in
`qram/qramtemplate/buckdatacell.py`. Given the address width, it generates a
router tree and attaches the auxiliary qubits required by the QRAM routing
procedure.

| Register role | Meaning |
| --- | --- |
| `address` | the query address, either a basis state or a superposition |
| `bus` | the readout qubit that carries the selected data bit |
| `incident` | a work qubit used during bucket-brigade routing |
| `router_*` | router nodes and leaves that guide the bus through the tree |
| `router_*_data` | leaf data qubits used by CZ operations to encode classical data |

The first example prepares the address register in a uniform superposition, so
one statevector simulation covers all four addresses at once.


In [ ]:
bucket_circuit, bucket_qram, address_qreg, bus_qreg = build_bucket_qram(
    n_address_qubits,
    memory_data,
    mode="native",
    address_state="uniform",
)

summary = resource_summary(bucket_circuit)
print("Circuit resource summary for the native bucket-brigade QRAM")
display(pd.DataFrame([summary]).drop(columns=["ops"]))
print("Gate counts in the generated QRAM circuit")
display(pd.DataFrame([{"gate": gate, "count": count} for gate, count in summary["ops"].items()]))

register_rows = []
for qreg in bucket_circuit.qregs:
    if qreg.name == "address":
        role = "query address"
    elif qreg.name == "bus":
        role = "readout bus"
    elif qreg.name == "incident":
        role = "routing work qubit"
    elif qreg.name.endswith("_data"):
        role = "leaf data cell"
    elif "router" in qreg.name:
        role = "router tree node"
    else:
        role = "other"
    register_rows.append({"register": qreg.name, "size": qreg.size, "role": role})

print("Quantum registers introduced by the QRAM construction")
display(pd.DataFrame(register_rows))
print("Text drawing of the generated QRAM circuit")
print(bucket_circuit.draw(output="text", fold=120))


## 3. Simulate QRAM Readout

The ideal one-bit QRAM transformation is:

$$
\sum_a \alpha_a \lvert a\rangle \lvert 0\rangle_{\mathrm{bus}}
\longrightarrow
\sum_a \alpha_a \lvert a\rangle \lvert d_a\rangle_{\mathrm{bus}}
$$

Because the address register is uniform, each address should appear with
probability `1/4`. The table below marginalizes the full statevector down to
`address + bus`, leaving the router and data ancillas out of the display.


In [ ]:
readout = logical_readout_probabilities(bucket_circuit, address_qreg, bus_qreg)

expected = expected_table(addresses, memory_data, probability=1 / len(addresses))
comparison = expected.merge(
    readout,
    left_on=["address", "expected_bus"],
    right_on=["address", "bus"],
    how="left",
)
comparison["probability"] = comparison["probability"].fillna(0.0)
comparison["abs_delta"] = (
    comparison["probability"] - comparison["expected_probability"]
).abs()

print("Statevector readout check against the expected memory table")
display(comparison[["address", "expected_bus", "expected_probability", "probability", "abs_delta"]])
print(f"L1 error on expected rows: {comparison['abs_delta'].sum():.3e}")


## 4. Query One Classical Address

The previous cell verifies a superposition of addresses. For a more direct
readout example, this cell prepares the classical address `10`. Since the memory
table stores `10 -> 1`, the final bus qubit should be `1` with probability 1.


In [ ]:
query_address = "10"
query_circuit, _, query_address_qreg, query_bus_qreg = build_bucket_qram(
    n_address_qubits,
    memory_data,
    mode="native",
    address_state=query_address,
)

query_readout = logical_readout_probabilities(query_circuit, query_address_qreg, query_bus_qreg)
print(f"Readout distribution for the classical query address {query_address}")
display(query_readout)


## 5. Compare CSWAP Implementations

Controlled-SWAP is the key routing primitive in bucket-brigade QRAM. The source
tree exposes three implementation paths:

| Mode | Source behavior |
| --- | --- |
| `native` | use Qiskit's native `cswap` gate |
| `cswap_decompose` | expand CSWAP into an `rx/rz/cz` sequence |
| `subspace_decompose` | use a shorter subspace decomposition sequence |

The next cell compares the resource cost of the three modes on the same
2-address QRAM instance and checks that the readout probabilities remain close
to the ideal table.


In [ ]:
mode_rows = []
readout_rows = []

for mode in ["native", "cswap_decompose", "subspace_decompose"]:
    circuit, _, a_qreg, b_qreg = build_bucket_qram(
        n_address_qubits,
        memory_data,
        mode=mode,
        address_state="uniform",
    )
    summary = resource_summary(circuit)
    mode_rows.append(
        {
            "mode": mode,
            "qubits": summary["qubits"],
            "depth": summary["depth"],
            "size": summary["size"],
            "swap_depth": summary["swap_depth"],
            "cswap_or_cz_depth": summary["cswap_or_cz_depth"],
            "ops": summary["ops"],
        }
    )

    mode_readout = logical_readout_probabilities(circuit, a_qreg, b_qreg)
    mode_comparison = expected.merge(
        mode_readout,
        left_on=["address", "expected_bus"],
        right_on=["address", "bus"],
        how="left",
    )
    mode_comparison["probability"] = mode_comparison["probability"].fillna(0.0)
    mode_comparison["abs_delta"] = (
        mode_comparison["probability"] - mode_comparison["expected_probability"]
    ).abs()
    readout_rows.append(
        {
            "mode": mode,
            "max_abs_delta": mode_comparison["abs_delta"].max(),
            "l1_error": mode_comparison["abs_delta"].sum(),
        }
    )

mode_resource = pd.DataFrame(mode_rows).drop(columns=["ops"])
mode_errors = pd.DataFrame(readout_rows)
mode_labels = mode_resource["mode"].str.replace("_", "\n")
mode_colors = ["#4E79A7", "#F28E2B", "#59A14F"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.3), constrained_layout=True)

resource_metrics = ["depth", "size", "swap_depth", "cswap_or_cz_depth"]
x = np.arange(len(mode_resource))
bar_width = 0.18
for metric_idx, metric in enumerate(resource_metrics):
    axes[0].bar(
        x + (metric_idx - 1.5) * bar_width,
        mode_resource[metric],
        bar_width,
        label=metric.replace("_", " "),
    )
axes[0].set_xticks(x)
axes[0].set_xticklabels(mode_labels)
axes[0].set_ylabel("count")
axes[0].set_title("Resource cost by CSWAP mode")
axes[0].legend(fontsize=8, frameon=False)
axes[0].grid(True, axis="y", alpha=0.3)

scatter = axes[1].scatter(
    mode_resource["swap_depth"],
    mode_resource["cswap_or_cz_depth"],
    s=np.maximum(mode_resource["size"], 1) * 24,
    c=mode_resource["depth"],
    cmap="viridis",
    edgecolors="black",
    linewidths=0.7,
)
for _, row in mode_resource.iterrows():
    axes[1].annotate(
        row["mode"].replace("_", "\n"),
        (row["swap_depth"], row["cswap_or_cz_depth"]),
        textcoords="offset points",
        xytext=(6, 5),
        fontsize=8,
    )
axes[1].set_xlabel("SWAP depth")
axes[1].set_ylabel("CSWAP/CZ depth")
axes[1].set_title("Depth trade-off scatter")
axes[1].grid(True, alpha=0.3)
fig.colorbar(scatter, ax=axes[1], label="circuit depth")

error_metrics = ["max_abs_delta", "l1_error"]
for metric_idx, metric in enumerate(error_metrics):
    axes[2].bar(
        x + (metric_idx - 0.5) * 0.32,
        mode_errors[metric],
        0.32,
        label=metric.replace("_", " "),
        color=["#76B7B2", "#E15759"][metric_idx],
        edgecolor="black",
        linewidth=0.5,
    )
axes[2].set_xticks(x)
axes[2].set_xticklabels(mode_labels)
axes[2].set_ylabel("absolute probability error")
axes[2].set_title("Readout error remains near zero")
axes[2].legend(fontsize=8, frameon=False)
axes[2].grid(True, axis="y", alpha=0.3)
axes[2].set_ylim(0, max(float(mode_errors[error_metrics].to_numpy().max()) * 1.2, 1e-12))
axes[2].ticklabel_format(axis="y", style="sci", scilimits=(0, 0))

plt.show()

lowest_depth = mode_resource.loc[mode_resource["depth"].idxmin(), "mode"]
smallest_size = mode_resource.loc[mode_resource["size"].idxmin(), "mode"]
largest_l1 = mode_errors["l1_error"].max()
print(f"Lowest-depth mode: {lowest_depth}; smallest circuit-size mode: {smallest_size}.")
print(f"Largest L1 readout error across modes: {largest_l1:.3e}.")


## 6. Optional: Noisy Readout With Qiskit Aer

The previous sections use exact statevector simulation. A lightweight noisy
experiment is also useful, but it requires two extra choices:

1. the QRAM circuit must include measurements, because Aer shot simulation
   returns classical counts;
2. the noise model should be attached to gates that the simulator executes.

This example uses the `subspace_decompose` circuit because it is already
expressed with one- and two-qubit gates (`rx`, `rz`, `cz`, `swap`) instead of
native three-qubit `cswap` gates. The noise model below is intentionally simple:
small depolarizing errors on one- and two-qubit gates plus a symmetric readout
error. It is a tutorial model, not a calibrated hardware model.

The default parameters used below are:

| Parameter | Value | Applied to |
| --- | ---: | --- |
| `p1` | `0.001` | one-qubit gate depolarizing error |
| `p2` | `0.01` | two-qubit gate depolarizing error |
| `p_readout` | `0.02` | symmetric measurement readout error |
| `aer_seed` | `20` | reproducible shot and noise sampling |


In [ ]:
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, ReadoutError, depolarizing_error


def make_tutorial_noise_model(p1=0.001, p2=0.01, p_readout=0.02):
    noise_model = NoiseModel()
    noise_model.add_all_qubit_quantum_error(
        depolarizing_error(p1, 1),
        ["x", "h", "rx", "rz"],
    )
    noise_model.add_all_qubit_quantum_error(
        depolarizing_error(p2, 2),
        ["cz", "cx", "swap"],
    )
    noise_model.add_all_qubit_readout_error(
        ReadoutError([[1 - p_readout, p_readout], [p_readout, 1 - p_readout]])
    )
    return noise_model


def parse_aer_counts(counts):
    rows = []
    total = sum(counts.values())
    for bitstring, shots in counts.items():
        compact = bitstring.replace(" ", "")
        # Qiskit prints classical registers in reverse register order here:
        # c_bus appears before c_address, and c_address bits are reversed.
        bus = compact[0]
        address = compact[1:][::-1]
        rows.append(
            {
                "address": address,
                "bus": bus,
                "shots": shots,
                "probability": shots / total,
            }
        )
    return (
        pd.DataFrame(rows)
        .groupby(["address", "bus"], as_index=False)[["shots", "probability"]]
        .sum()
        .sort_values(["address", "bus"])
        .reset_index(drop=True)
    )


def readout_success_probability(distribution, data):
    expected_bus = {address: str(bit) for address, bit in zip(addresses, data)}
    return float(
        distribution[
            distribution.apply(lambda row: expected_bus[row["address"]] == row["bus"], axis=1)
        ]["probability"].sum()
    )


aer_circuit, _, _, _ = build_bucket_qram(
    n_address_qubits,
    memory_data,
    mode="subspace_decompose",
    address_state="uniform",
    include_measurements=True,
)

basis_gates = ["rx", "rz", "x", "h", "cz", "cx", "swap", "measure"]
aer_circuit = transpile(aer_circuit, basis_gates=basis_gates, optimization_level=0)

shots = 4096
aer_seed = 20
ideal_backend = AerSimulator(seed_simulator=aer_seed)
noisy_backend = AerSimulator(
    noise_model=make_tutorial_noise_model(),
    seed_simulator=aer_seed,
)

ideal_counts = ideal_backend.run(aer_circuit, shots=shots).result().get_counts()
noisy_counts = noisy_backend.run(aer_circuit, shots=shots).result().get_counts()

ideal_distribution = parse_aer_counts(ideal_counts)
noisy_distribution = parse_aer_counts(noisy_counts)

success_rows = pd.DataFrame(
    [
        {
            "simulation": "Aer ideal shots",
            "success_probability": readout_success_probability(ideal_distribution, memory_data),
        },
        {
            "simulation": "Aer noisy shots",
            "success_probability": readout_success_probability(noisy_distribution, memory_data),
        },
    ]
)

aer_summary = resource_summary(aer_circuit)
print(
    "Measured Aer circuit: "
    f"{aer_summary['qubits']} qubits, depth {aer_summary['depth']}, "
    f"size {aer_summary['size']}."
)

plot_distributions = pd.concat(
    [
        ideal_distribution.assign(simulation="ideal"),
        noisy_distribution.assign(simulation="noisy"),
    ],
    ignore_index=True,
)
plot_distributions["outcome"] = (
    "|" + plot_distributions["address"] + "> -> bus " + plot_distributions["bus"]
)
outcome_order = sorted(plot_distributions["outcome"].unique())
distribution_by_sim = {
    simulation: (
        plot_distributions[plot_distributions["simulation"] == simulation]
        .set_index("outcome")
        .reindex(outcome_order)
        .fillna({"shots": 0, "probability": 0.0})
    )
    for simulation in ["ideal", "noisy"]
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), constrained_layout=True)
x = np.arange(len(outcome_order))
width = 0.38
axes[0].bar(
    x - width / 2,
    distribution_by_sim["ideal"]["probability"],
    width,
    label="ideal",
    color="#9ECAE1",
    edgecolor="black",
    linewidth=0.5,
)
axes[0].bar(
    x + width / 2,
    distribution_by_sim["noisy"]["probability"],
    width,
    label="noisy",
    color="#FB6A4A",
    edgecolor="black",
    linewidth=0.5,
)
axes[0].set_xticks(x)
axes[0].set_xticklabels(outcome_order, rotation=35, ha="right")
axes[0].set_ylabel("shot probability")
axes[0].set_title("Aer readout distribution")
axes[0].legend(frameon=False)
axes[0].grid(True, axis="y", alpha=0.3)

axes[1].bar(
    success_rows["simulation"].str.replace("Aer ", "").str.replace(" shots", ""),
    success_rows["success_probability"],
    color=["#4E79A7", "#E15759"],
    edgecolor="black",
    linewidth=0.6,
)
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel("success probability")
axes[1].set_title("Readout success under tutorial noise")
axes[1].grid(True, axis="y", alpha=0.3)
plt.show()

for _, row in success_rows.iterrows():
    print(f"{row['simulation']}: success probability {row['success_probability']:.3f}")


## 7. Inspect Scaling Data Already In The Repository

Large QRAM statevector or density-matrix simulations become expensive quickly.
To keep this tutorial lightweight and reproducible, this section uses the
resource CSV files already checked into the repository. They show how qubit
count, SWAP/CSWAP depth, and mapping resources grow with the QRAM level.

The files are under `realdata/data/depth_gate/`:

| File | Meaning |
| --- | --- |
| `counts_data.csv` | resource statistics for the data-load QRAM circuit |
| `counts_nodedataload.csv` | resource statistics without data loading |
| `h_tree.csv` | teleportation and SWAP statistics for H-tree mapping |

In these CSV files, `counts_data.csv` and `counts_nodedataload.csv` use the
same qubit allocation for each QRAM level, so their qubit-count curves overlap.
The difference appears in depth and gate counts rather than in `num_qubits`.


In [ ]:
depth_dir = repo_root / "realdata" / "data" / "depth_gate"
counts_data = pd.read_csv(depth_dir / "counts_data.csv")
counts_noload = pd.read_csv(depth_dir / "counts_nodedataload.csv")
h_tree = pd.read_csv(depth_dir / "h_tree.csv")

print(
    "Loaded resource curves: "
    f"{len(counts_data)} data-load levels, "
    f"{len(counts_noload)} no-data-load levels, "
    f"{len(h_tree)} H-tree levels."
)

fig, axes = plt.subplots(2, 2, figsize=(14, 8.5), constrained_layout=True)
axes = axes.ravel()

axes[0].scatter(counts_data["level"], counts_data["num_qubits"], s=70, label="data load", color="#4E79A7")
axes[0].plot(counts_data["level"], counts_data["num_qubits"], color="#4E79A7", alpha=0.65)
axes[0].scatter(counts_noload["level"], counts_noload["num_qubits"], s=45, marker="s", label="no data load", color="#F28E2B")
axes[0].plot(counts_noload["level"], counts_noload["num_qubits"], color="#F28E2B", alpha=0.65)
axes[0].set_xlabel("QRAM level")
axes[0].set_ylabel("qubits")
axes[0].set_title("Qubit-count scaling")
axes[0].legend(frameon=False)
axes[0].grid(True, alpha=0.3)

for frame, label, color in [
    (counts_data, "data load", "#4E79A7"),
    (counts_noload, "no data load", "#F28E2B"),
]:
    axes[1].plot(frame["level"], frame["cswap_depth"], marker="o", color=color, label=f"{label}: CSWAP/CZ")
    axes[1].plot(frame["level"], frame["swap_depth"], marker="s", color=color, linestyle="--", label=f"{label}: SWAP")
axes[1].set_xlabel("QRAM level")
axes[1].set_ylabel("depth")
axes[1].set_title("Depth scaling")
axes[1].legend(fontsize=8, frameon=False)
axes[1].grid(True, alpha=0.3)

axes[2].plot(counts_data["level"], counts_data["cswap_count"], marker="o", label="data load: CSWAP/CZ", color="#4E79A7")
axes[2].plot(counts_data["level"], counts_data["swap_count"], marker="s", label="data load: SWAP", color="#76B7B2")
axes[2].plot(counts_noload["level"], counts_noload["cswap_count"], marker="o", linestyle="--", label="no data load: CSWAP/CZ", color="#F28E2B")
axes[2].plot(counts_noload["level"], counts_noload["swap_count"], marker="s", linestyle="--", label="no data load: SWAP", color="#E15759")
axes[2].set_xlabel("QRAM level")
axes[2].set_ylabel("gate count")
axes[2].set_title("Routing-gate count scaling")
axes[2].legend(fontsize=8, frameon=False)
axes[2].grid(True, alpha=0.3)

h_scatter = axes[3].scatter(
    h_tree["swap_count"],
    h_tree["tele_count"],
    c=h_tree["level"],
    s=80 + 8 * (h_tree["tele_depth"] + h_tree["swap_depth"]),
    cmap="viridis",
    edgecolors="black",
    linewidths=0.7,
)
for _, row in h_tree.iterrows():
    axes[3].annotate(
        f"L{int(row['level'])}",
        (row["swap_count"], row["tele_count"]),
        textcoords="offset points",
        xytext=(5, 4),
        fontsize=8,
    )
axes[3].set_xlabel("H-tree SWAP count")
axes[3].set_ylabel("H-tree teleportation count")
axes[3].set_title("H-tree mapping trade-off")
axes[3].grid(True, alpha=0.3)
fig.colorbar(h_scatter, ax=axes[3], label="QRAM level")

plt.show()


## 8. Plot Processed Experimental Data

The Aer example above uses an artificial noise model. This section switches to
processed experimental data included with the project. The code below only
loads preprocessed artifacts and draws figures; it does not submit new hardware
jobs or rerun the original calibration pipeline.

To keep the tutorial environment lightweight, the plots use artifacts that can
be loaded directly without the full `qutip`-dependent analysis notebooks:

| Artifact used by the next cell | What the tutorial extracts from it |
| --- | --- |
| teleportation fidelity records | single-qubit teleportation fidelity for six input states |
| processed fidelity sweep | layer-dependent QRAM fidelity curve and experimental reference points |
| error-mitigation records | query fidelity and accepted-data portion before/after selection |


In [ ]:
import pickle

realdata_dir = repo_root / "realdata" / "data"

processed_fidelity = pd.read_csv(realdata_dir / "processed_fidelity.csv")
mitigation_data = pd.read_csv(realdata_dir / "error_mitigation_pauli_tracking_fidelity.csv")
with open(realdata_dir / "Fig2b_data_pauli_tracking.pkl", "rb") as file:
    teleportation_data = pickle.load(file)

mitigation_data["address"] = mitigation_data["address"].astype(str)
mitigation_data["select"] = mitigation_data["select"].astype(str)

state_order = ["0", "1", "+", "-", "+i", "-i"]
teleportation_summary = pd.DataFrame(
    [
        {
            "state": f"|{state}>",
            "samples": len(teleportation_data[state]),
            "fidelity_mean": np.mean(teleportation_data[state]),
            "fidelity_std": np.std(teleportation_data[state]),
        }
        for state in state_order
    ]
)

main_selects = ["False", "True"]
mitigation_summary = (
    mitigation_data[mitigation_data["select"].isin(main_selects)]
    .groupby(["address", "select"], as_index=False)
    .agg(
        samples=("fidelity", "count"),
        fidelity_mean=("fidelity", "mean"),
        fidelity_std=("fidelity", "std"),
        valid_portion_mean=("valid_portion", "mean"),
        valid_portion_std=("valid_portion", "std"),
    )
)

print(
    "Loaded processed artifacts: "
    f"{sum(len(teleportation_data[state]) for state in state_order)} teleportation samples, "
    f"{len(processed_fidelity)} fidelity-sweep points, "
    f"{len(mitigation_data)} mitigation records."
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)

axes[0].bar(
    teleportation_summary["state"],
    teleportation_summary["samples"],
    color="#4E79A7",
    edgecolor="black",
    linewidth=0.6,
)
axes[0].set_xlabel("input state")
axes[0].set_ylabel("samples")
axes[0].set_title("Teleportation sample coverage")
axes[0].grid(True, axis="y", alpha=0.3)

coverage_scatter = axes[1].scatter(
    processed_fidelity["pp"],
    processed_fidelity["layer"],
    c=processed_fidelity["fqram"],
    s=75,
    cmap="viridis",
    edgecolors="black",
    linewidths=0.4,
)
axes[1].set_xscale("log")
axes[1].set_xlabel("physical Pauli error pp")
axes[1].set_ylabel("QRAM layer")
axes[1].set_title("Processed fidelity sweep coverage")
axes[1].grid(True, which="both", alpha=0.25)
fig.colorbar(coverage_scatter, ax=axes[1], label="F_QRAM")

sample_counts = (
    mitigation_summary.pivot(index="address", columns="select", values="samples")
    .reindex(sorted(mitigation_summary["address"].unique()))
)
address_x = np.arange(len(sample_counts))
width = 0.38
axes[2].bar(
    address_x - width / 2,
    sample_counts["False"],
    width,
    label="raw",
    color="#9ECAE1",
    edgecolor="black",
    linewidth=0.5,
)
axes[2].bar(
    address_x + width / 2,
    sample_counts["True"],
    width,
    label="selected",
    color="#FB6A4A",
    edgecolor="black",
    linewidth=0.5,
)
axes[2].set_xticks(address_x)
axes[2].set_xticklabels([f"|{address}>" for address in sample_counts.index], rotation=30)
axes[2].set_ylabel("records")
axes[2].set_title("Mitigation records by address")
axes[2].legend(frameon=False)
axes[2].grid(True, axis="y", alpha=0.3)

plt.show()


The figure below should be read as a compact summary of the processed
experimental artifacts:

| Panel | How to read it |
| --- | --- |
| Teleportation fidelity | bars show the mean fidelity for six input states; error bars show the standard deviation across processed samples |
| Processed fidelity sweep | curves show the layer-dependent QRAM infidelity `1 - F` versus physical Pauli error `pp`; star markers are the experimental reference points included with the processed data |
| Selection vs raw query fidelity | blue bars are raw query fidelities and orange bars are fidelities after selection/error mitigation |
| Accepted portion after selection | green bars show the fraction of data retained by the selection rule; the dashed line marks the raw-data baseline where all data are retained |

The main message is that selection improves query fidelity across the displayed
addresses, but this improvement comes with a data-retention cost. The fidelity
sweep also shows the expected trend that larger QRAM layers and higher physical
error rates lead to larger QRAM infidelity.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
axes = axes.ravel()

# Teleportation fidelity for six input states.
tele_x = np.arange(len(teleportation_summary))
axes[0].bar(
    tele_x,
    teleportation_summary["fidelity_mean"],
    yerr=teleportation_summary["fidelity_std"],
    capsize=4,
    color=["#4E79A7", "#F28E2B", "#59A14F", "#E15759", "#76B7B2", "#EDC948"],
    edgecolor="black",
    linewidth=0.6,
)
axes[0].set_xticks(tele_x)
axes[0].set_xticklabels(teleportation_summary["state"])
axes[0].set_ylim(0.975, 1.0)
axes[0].set_ylabel("fidelity")
axes[0].set_title("Teleportation fidelity")
axes[0].grid(True, axis="y", alpha=0.3)

# Processed fidelity sweep, with the experimental points marked by stars.
cmap = plt.get_cmap("viridis", processed_fidelity["layer"].nunique())
for color_idx, layer in enumerate(sorted(processed_fidelity["layer"].unique())):
    layer_data = processed_fidelity[processed_fidelity["layer"] == layer].sort_values("pp")
    axes[1].plot(
        layer_data["pp"],
        1 - layer_data["fqram"],
        marker="o",
        markersize=4,
        linewidth=1.5,
        color=cmap(color_idx),
        label=f"L{layer}",
    )

experimental_points = pd.DataFrame(
    [
        {"layer": 2, "pp": 0.0038, "fqram": 0.595},
        {"layer": 3, "pp": 0.0038, "fqram": 0.200},
    ]
)
for _, row in experimental_points.iterrows():
    color_idx = int(row["layer"] - processed_fidelity["layer"].min())
    axes[1].scatter(
        row["pp"],
        1 - row["fqram"],
        marker="*",
        s=180,
        color=cmap(color_idx),
        edgecolor="black",
        linewidth=0.7,
        zorder=5,
    )
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel("physical Pauli error pp")
axes[1].set_ylabel("QRAM infidelity 1 - F")
axes[1].set_title("Processed fidelity sweep (stars: experiments)")
axes[1].legend(ncol=4, fontsize=8, frameon=False)
axes[1].grid(True, which="both", alpha=0.25)

# Query fidelity before and after selection/error mitigation.
address_order = list(dict.fromkeys(mitigation_data["address"]))
summary_by_select = {
    select: (
        mitigation_summary[mitigation_summary["select"] == select]
        .set_index("address")
        .reindex(address_order)
    )
    for select in main_selects
}
x = np.arange(len(address_order))
width = 0.38
axes[2].bar(
    x - width / 2,
    summary_by_select["False"]["fidelity_mean"],
    width,
    yerr=summary_by_select["False"]["fidelity_std"],
    capsize=3,
    label="raw",
    color="#9ecae1",
    edgecolor="black",
    linewidth=0.5,
)
axes[2].bar(
    x + width / 2,
    summary_by_select["True"]["fidelity_mean"],
    width,
    yerr=summary_by_select["True"]["fidelity_std"],
    capsize=3,
    label="selected",
    color="#fb6a4a",
    edgecolor="black",
    linewidth=0.5,
)
axes[2].set_xticks(x)
axes[2].set_xticklabels([f"|{address}>" for address in address_order], rotation=30)
axes[2].set_ylim(0.5, 0.9)
axes[2].set_ylabel("query fidelity")
axes[2].set_title("Selection vs raw query fidelity")
axes[2].legend(frameon=False)
axes[2].grid(True, axis="y", alpha=0.3)

# Selection improves fidelity while retaining only a portion of the data.
selected_summary = summary_by_select["True"]
axes[3].bar(
    x,
    selected_summary["valid_portion_mean"],
    yerr=selected_summary["valid_portion_std"],
    capsize=3,
    color="#74c476",
    edgecolor="black",
    linewidth=0.5,
)
axes[3].axhline(1.0, color="#525252", linestyle="--", linewidth=1, label="raw data portion")
axes[3].set_xticks(x)
axes[3].set_xticklabels([f"|{address}>" for address in address_order], rotation=30)
axes[3].set_ylim(0.55, 1.05)
axes[3].set_ylabel("accepted data portion")
axes[3].set_title("Accepted portion after selection")
axes[3].legend(frameon=False)
axes[3].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


## What This Notebook Demonstrated

1. Built a small bucket-brigade QRAM instance with 2 address qubits, 4 memory
   cells, and 1 bus qubit.
2. Explained the roles of `address`, `bus`, `incident`, `router_*`, and
   `router_*_data` registers.
3. Verified QRAM readout on an address superposition with local `Statevector`
   simulation.
4. Demonstrated a direct classical query for address `10`.
5. Compared native CSWAP, full CSWAP decomposition, and subspace decomposition
   in terms of depth, gate counts, and statevector probability error.
6. Ran a lightweight optional Qiskit Aer experiment with depolarizing and
   readout noise, reporting the readout success probability.
7. Used repository CSV files to inspect how QRAM resources scale with the
   circuit level.
8. Loaded processed experimental datasets from `realdata/data/` and plotted
   teleportation fidelity, QRAM infidelity scaling, query fidelity, and the
   accepted-data portion after selection.

A natural extension is to turn the GPU density-matrix simulations under
`simulations/` or the full `qutip`-dependent real-data notebooks into a second
notebook. This first tutorial stays lightweight, fast, and easy to review.
